### **Library & Package Imports**

In [ ]:
import requests, json

import os

from deep_translator import GoogleTranslator
import deepl

### **Hard-Coded Data**

##### **ISO code for each Wikipedia language edition used**

In [ ]:
edition_codes = {
    "en" : "English",    "ru" : "Russian",    "es" : "Spanish",    "vi" : "Vietnamese",
    "hi" : "Hindi",    "ga" : "Irish",    "af" : "Afrikaans",    "ar" : "Arabic",
    "fa" : "Persian",    "zh" : "Chinese",    "de" : "German",    "be" : "Belarusian",
    "fr" : "French",    "sw" : "Swahili",    "bn" : "Bengali",    "sq" : "Albanian",
    "ur" : "Urdu",    "it" : "Italian",    "pl" : "Polish",    "jam" : "Jamaican Patois",
    "am" : "Amharic", "ja" : "Japanese", "arz" : "Egyptian Arabic", "he" : "Hebrew",
    "si" : "Sinhala", "ta" : "Tamil", "zu" : "Zulu"
}

##### **Wikipedia URL title for each article edition**

In [ ]:
article_titles = {
    "John_F_Kennedy" : [ ("en", "John_F._Kennedy"),
                        ("ru", "Кеннеди,_Джон_Фицджералд"), 
                        ("es", "John_F._Kennedy"),
                        ("vi", "John_F._Kennedy") ],
    "Queen_Elizabeth_II" : [ ("en", "Elizabeth_II"),
                            ("hi", "एलिज़ाबेथ_द्वितीय"), 
                            ("ga", "Eilís_II_na_Ríochta_Aontaithe"),
                            ("af", "Elizabeth_II_van_die_Verenigde_Koninkryk") ],
    "Saddam_Hussein" : [ ("en", "Saddam_Hussein"),
                       ("ar", "صدام_حسين"),
                       ("fa", "صدام_حسین") ],
    "Mikhail_Gorbachev" : [ ("en", "Mikhail_Gorbachev"),
                           ("ru", "Горбачёв,_Михаил_Сергеевич"),
                           ("zh", "米哈伊尔·戈尔巴乔夫"),
                           ("de", "Michail_Sergejewitsch_Gorbatschow"),
                           ("be", "Міхаіл_Сяргеевіч_Гарбачоў") ],
    "Che_Guevara" : [ ("en", "Che_Guevara"),
                    ("es", "Che_Guevara"),
                    ("fr", "Che_Guevara"),
                    ("sw", "Che_Guevara") ],
    "Mother_Teresa" : [ ("bn", "মাদার_টেরিজা"),
                    ("sq", "Nënë_Tereza"),
                    ("ur", "مدر_ٹریسا"),
                    ("en", "Mother_Teresa") ],
    "Pope_John_Paul_II" : [ ("it", "Papa_Giovanni_Paolo_II"),
                           ("ar", "يوحنا_بولس_الثاني"),
                           ("pl", "Jan_Paweł_II"),
                           ("en", "Pope_John_Paul_II")],
    "Jean-Paul_Sartre" : [ ("fr", "Jean-Paul_Sartre"),
                         ("ru", "Сартр,_Жан-Поль"),
                         ("en", "Jean-Paul_Sartre") ],
    "Bob_Marley" : [ ("jam", "Bab_Maali"),
                    ("am", "ቦብ_ማርሊ"),
                    ("ja", "ボブ・マーリー"),
                    ("en", "Bob_Marley") ],
    "Umm_Kulthum" : [ ("arz", "ام_كلثوم"),
                    ("he", "אום_כולתום"),
                    ("ar", "أم_كلثوم_(مطربة)"),
                    ("en", "Umm_Kulthum") ],
    "Sirimavo_Bandaranaike" : [ ("si", "සිරිමාවෝ_බණ්ඩාරනායක"),
                              ("ta", "சிறிமாவோ_பண்டாரநாயக்கா"),
                              ("en", "Sirimavo_Bandaranaike") ],
    "Nelson_Mandela" : [ ("af", "Nelson_Mandela"),
                       ("zu", "Nelson_Mandela"),
                       ("ru", "Мандела,_Нельсон"),
                       ("en", "Nelson_Mandela")]
}

### **File Reading & Writing**

##### **Saving the (extracted) article content to a text file**

In [ ]:
def save_content_to_file (content : str, file_path: str):
    """
    This function takes the content of a Wikipedia article, and saves it to a text file.
    
    :content: The string that contains (what is assumed to be) the article content.
    :file_path: The file path where the content should be saved to.
    If the file already exists, you will have the option to (not) overwrite it.
    """
    answer = "Y"

    if os.path.isfile(file_path) == True:
        answer = input(f'There is already a file at "{file_path}". Is it ok to overwrite it? (Y/N) ')

    if len(file_path) <= 4 or file_path[-4:] != ".txt":
        print("The file path given must be for a text file.")
    elif answer in ("Y", "y"):
        try:
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(content)
                # Using the 'with' keyword removes the need to manually close the file
            print(f"The article content has been saved to '{file_path}'.")
        except:
            print("The article content could not be saved due to an error.")
    else:
        print("The article content was not saved to avoid overwriting the existing file.")

##### **Reading from article content text files**

In [ ]:
def read_text_file (file_path : str):
    """
    This function returns the contents of a text file as a string.
    
    :file_path: The file path of the text file.
    
    Returns the file text as a string, or None if the file could not be opened.
    """
    text = ""
    if os.path.isfile(file_path) == False or len(file_path) <= 4 or file_path[-4:] != ".txt":
        print(f"'{file_path}' is not a text file.")
        return None
    else:
        text = ""
        try:
            with open(file_path, "r", encoding="utf-8") as file:
                text = file.read()
                # Using the 'with' keyword removes the need to manually close the file
        except:
            # If the file could not be opened/read
            print(f"The file at '{file_path}' could not be read.")
            text = None
        
        return text

##### **Saving article content in bulk**

In [ ]:
def bulk_save_content (content_list : list, file_path_list : list):
    """
    This function saves the content of multiple wikipedia articles to seperate text files.
    
    :content_list: A list of strings containing (what is assumed to be) the article content.
    :file_path_list: A list of file paths denoting where each article's content should be saved to.
    If any file path points to an existing file, you will have the option to (not) overwrite it.
    """
    if len(content_list) > len(file_path_list):
        print("Error - At least one file path must be provided for each content string.")
    else:
        for content, file_path in zip(content_list, file_path_list):
            save_content_to_file(content, file_path)

### **Article Content Extraction**

##### **Extracting the article content (using the TextExtracts API)**

In [ ]:
def extract_article_content (edition_code : str, article_title : str):
    """
    This function retrieves the plaintext content of a Wikipedia article, using MediaWiki's TextExtracts API.
    
    :edition_code: The code for the target Wikipedia language edition.
    :article_title: The title of the target Wikipedia article.
    Note that the same article will likely have a different title in different editions.
    
    Returns the article content as a string, or None if the request to the API was unsuccessful. 
    """
    url = "https://" + edition_code + ".wikipedia.org/w/api.php?action=query&prop=extracts&titles=" + article_title + "&explaintext=1&format=json"

    headers = {"User-Agent" : "Wikipedia Article Content Extraction"}
    # If the user agent is not set, or it is overly generic, then the response will be a 403 error.
    # This seems to be MediaWiki's automated system to stop bots.

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        content = data['query']['pages']
        content = content[list(content.keys())[0]]['extract']
        print(f"The content of the article '{article_title}' ({edition_code}) was successfully retrieved.")
        return content
    else:
        print(f"The content of the article '{article_title}' ({edition_code}) could not be retrieved (error status code: {response.status_code}).")
        return None

# An example TextExtracts API endpoint (URL):
# https://en.wikipedia.org/w/api.php?action=query&prop=extracts&titles=John_F._Kennedy&explaintext=1&format=json

##### **Extracting article content in bulk**

In [ ]:
def bulk_extract_article_content (article_titles):
    """
    This function retrieves the plaintext content of multiple Wikipedia articles / article editions.
    
    :article_titles: A list of tuples, each in the format (language edition code, article title).
    Note that the same article will likely have a different title in different language editions.
    
    Returns a dictionary in the format of { title & edition code : article content (as a string) }.
    """
    content_dict = dict()
    
    for article in article_titles:
        content = extract_article_content(article[0], article[1])
        if content != None:
            content_dict[article[1] + " " + article[0]] = content
            
    return content_dict

### **Article Content Translation**

##### **Translation (using the 'deep_translator' library)**

In [ ]:
def google_translate_text (text : str, source_lang : str = None):
    """
    This is function takes text in any language and translates it into English, using the deep_translator library.
    It does this using the GoogleTranslator package of the deep_translator library.
    
    :text: The text to be translated.
    :source_lang: (optional) The ISO code of the language that the text is in.
    By default (or if set to 'auto'), the text's language will be automatically detected.
    
    Returns a string containing the translated English text, or an empty string if an error occurs.
    """
    # The deep_translator library only works with a limited number (5000) of characters at a time.
    # To solve this, the untranslated text is split up into paragraphs before being translated.
    split_text = text.split("\n") # Splits up the text into paragraphs.
    
    # Translate and recombine all of the paragraphs
    print("Translating text, please wait...")
    try:
        if source_lang == None or source_lang == "auto":
            translation_list = GoogleTranslator(source='auto', target='en').translate_batch(split_text)
        else:
            translation_list = GoogleTranslator(source=source_lang, target='en').translate_batch(split_text)
    except Exception as e:
        print(f"Error - {e}")
        return ""
        
    result = ""
    for translation in translation_list:
        result += translation + "\n"

    return result

##### **Translation (using the DeepL Translator API)**

In [ ]:
def deepl_translate_text (text : str, source_lang : str, api_key : str):
    """
    This is function takes text in any language and translates it into English, using DeepL's API.
    A valid DeepL API key is required.
    For free-tier users, you will be able to see whether you have reached your monthly translation limit.
    
    :text: The text to be translated.
    :source_lang: The ISO code of the language that the text is in.
    :api_key: Your DeepL API key.
    
    Returns a string containing the translated English text, or an empty string if an error occurs.
    """
    try:    
        deepl_client = deepl.DeepLClient(api_key) # Validates the given DeepL API key
        
        usage = deepl_client.get_usage()
        if usage.any_limit_reached:
            # If the free-tier limit of 500,000 characters has been reached for this API key
            print('Your monthly DeepL translation limit (of 500,000 characters) has been reached.')
            return ""
        elif usage.character.valid:
            # If the free-tier limit has not been reached (but there is still a limit for this API key)
            print(f"{usage.character.count} characters used out of your {usage.character.limit} monthly limit.")
        
        # Confirmation message to prevent accidental translation
        answer = input(f'Are you sure you want to proceed with translation? (Y/N)')
        if answer in ("Y", "y"):
            print("Translating text, please wait...")
            result = deepl_client.translate_text(text, source_lang=source_lang.upper(), target_lang="EN")
            return result
        else:
            print("Translation cancelled.")
            return ""

    except Exception as e:
        print(f"Error - {e}")
        return ""

### **Code Execution Area**

##### **Extracting and saving the content of a singular article edition**

In [ ]:
single_content = extract_article_content("en", "John_F._Kennedy")

save_content_to_file(single_content, "Article_Text_English.txt")

##### **Extracting and saving the content for all editions of an article**

In [ ]:
bulk_content = bulk_extract_article_content(article_titles["John_F_Kennedy"])

file_path_list = [] # The list of file paths where extracted article content will be saved

for edition in ["English", "Russian", "Spanish", "Vietnamese"]:
    filename = "Article_Text_" + edition + ".txt"
    file_path_list.append(filename)
    
bulk_save_content( list(bulk_content.values()), file_path_list )

##### **Translating article content**

In [ ]:
### Using deep_translator
# print(google_translate_text(list(bulk_content.values())[1]))


### Using DeepL API
# print( deepl_translate_text(list(bulk_content.values())[1], "hi", "fake-api-key") )